In [58]:
# Run this if facing issue with transformers
# pip install torch torchvision torchaudio transformers

import pandas as pd
import numpy as np
import torch
import re
from transformers import BertForSequenceClassification, BertTokenizer, AutoTokenizer, AutoModelForSequenceClassification, pipeline, Trainer, TrainingArguments
from datasets import Dataset
from scipy.stats import loguniform
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
# from sklearn.model_selection import ParameterGrid
# import tiktoken

In [42]:
train_df = pd.read_csv('Data Updated/train_df.csv')
val_df = pd.read_csv('Data Updated/val_df.csv')
test_df = pd.read_csv('Data Updated/test_df.csv')

In [43]:
# Apply preprocessing to datasets
def clean_text(text):
    if isinstance(text, str):
        # Convert to lowercase
        text = text.lower()
        # Remove special characters and punctuation
        text = re.sub(r"[^\w\s]", "", text)
        # Remove extra spaces
        text = re.sub(r"\s+", " ", text).strip()
    else:
        text = ""
    return text

def preprocess_dataset(df):
    df['title'] = df['title'].apply(clean_text)
    df['sentiment_label'] = df['sentiment_label'].astype(int)
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)  # Shuffle dataset
    return df

train_df = preprocess_dataset(train_df)
val_df = preprocess_dataset(val_df)
test_df = preprocess_dataset(test_df)

In [44]:
# # REMOVE LATER - for testing labelling only
# train_df = train_df.head(600)
# val_df = val_df.head(200)
# test_df = test_df.head(200)

# Prepare df for labelling results
test_df_results = test_df.copy()

In [53]:
# Check distribution of labels
print(train_df['sentiment_label'].value_counts())
print(val_df['sentiment_label'].value_counts())
print(test_df['sentiment_label'].value_counts())

1    133131
0    128476
Name: sentiment_label, dtype: int64
1    32626
0    31459
Name: sentiment_label, dtype: int64
0    14673
1    13714
Name: sentiment_label, dtype: int64


# FinBERT (Fine Tuned for Sentiment Analysis)
FinBERT is a pre-trained text analysis model, trained on financial data. It labels financial data as either "positive", "neutral" or "negative" by assigning a probability to each class, and return the class with the highest probability.
<br> For our use case, we labelled sentiments using only '1' or '0' for the excess 3 day aggregated returns. Hence, we will first fine tune finBERT to label using 2 classifications on our training set, then using random search to search for the best parameters that will return the model with the highest F1 by evaluating its performance on the valuation set.

## BERT

## Yiyang/FinBERT TONE

## Prosus AI/FinBERT

In [ ]:
# If gpu available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load tokenizer and model
tokenizer = BertTokenizer.from_pretrained("ProsusAI/finbert")
model = BertForSequenceClassification.from_pretrained('ProsusAI/finbert', num_labels=2, ignore_mismatched_sizes=True).to(device)
model.config.problem_type = "single_label_classification" # To use finBERT for 2 labels

# Tokenize data
def preprocess_data(example):
    tokenized = tokenizer(example['title'], padding='max_length', max_length=128, truncation=True)
    tokenized['labels'] = example['sentiment_label']
    return tokenized

train_tokenized_df = Dataset.from_pandas(train_df).map(preprocess_data, batched=True)
train_tokenized_df.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

val_tokenized_df = Dataset.from_pandas(val_df).map(preprocess_data, batched=True)
val_tokenized_df.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

test_tokenized_df = Dataset.from_pandas(test_df).map(preprocess_data, batched=True)
test_tokenized_df.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

# Define random parameter space
random_param_space = {
    "learning_rate": loguniform(1e-6, 5e-5).rvs(50).tolist(),
    "batch_size": [16, 32],
    "num_epochs": [2, 3, 4],
    "threshold": np.random.uniform(0.05, 0.7, 50).tolist(),
    "weight_decay": [0.0, 0.01, 0.1]
}

num_random_samples = 50
random_param_combinations = [
    {
        "learning_rate": random.choice(random_param_space["learning_rate"]),
        "batch_size": random.choice(random_param_space["batch_size"]),
        "num_epochs": random.choice(random_param_space["num_epochs"]),
        "threshold": random.choice(random_param_space["threshold"]),
        "weight_decay": random.choice(random_param_space["weight_decay"]),
    }
    for _ in range(num_random_samples)
]

best_f1 = 0
best_params = None
best_threshold = None
best_model = None

# Define metrics
def compute_metrics(eval_pred):
    predictions, labels = eval_pred.predictions, eval_pred.label_ids
    predicted_probabilities = torch.softmax(torch.tensor(predictions), dim=1).numpy()

    # Apply threshold from params
    threshold = params["threshold"]
    predicted_labels = (predicted_probabilities[:, 1] >= threshold).astype(int)

    # Calculate metrics
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predicted_labels, average='binary')
    accuracy = accuracy_score(labels, predicted_labels)

    return {
        'eval_accuracy': accuracy,
        'eval_precision': precision,
        'eval_recall': recall,
        'eval_f1': f1
    }

# Perform random search
for params in random_param_combinations:
    print(f"Training with parameters: {params}")

    # Define training arguments
    training_args = TrainingArguments(
        output_dir='./results',
        evaluation_strategy='epoch',  # Evaluates at the end of each epoch
        save_strategy='epoch',  # Saves the model at the end of each epoch
        logging_dir='./logs',
        logging_steps=10,  # Log training loss every 10 steps
        logging_first_step=True,  # Log the first step to capture initial metrics
        per_device_train_batch_size=params['batch_size'],
        learning_rate=params['learning_rate'],
        num_train_epochs=params['num_epochs'],
        weight_decay=params['weight_decay'],
        load_best_model_at_end=True,
        metric_for_best_model='eval_f1',
        save_total_limit=1
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tokenized_df,
        eval_dataset=val_tokenized_df,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics
    )

    # Train and evaluate
    trainer.train()
    eval_metrics = trainer.evaluate()
    print("Validation metrics:", eval_metrics)

    # Track best model
    if 'eval_f1' in eval_metrics and eval_metrics['eval_f1'] > best_f1:
        best_f1 = eval_metrics['eval_f1']
        best_metrics = eval_metrics
        best_params = params
        best_threshold = params['threshold']
        best_model = trainer.model
        trainer.save_model("./finbert_best_model")

print("Best F1 Score:", best_f1)
print("Best Eval Metrics:", best_metrics)
print("Best Parameters:", best_params)
print("Best Threshold:", best_threshold)

Using device: cuda


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at ProsusAI/finbert and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([3, 768]) in the checkpoint and torch.Size([2, 768]) in the model instantiated
- classifier.bias: found shape torch.Size([3]) in the checkpoint and torch.Size([2]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/261607 [00:00<?, ? examples/s]

Map:   0%|          | 0/64085 [00:00<?, ? examples/s]

Map:   0%|          | 0/28387 [00:00<?, ? examples/s]

Training with parameters: {'learning_rate': 4.147589698454849e-06, 'batch_size': 32, 'num_epochs': 2, 'threshold': 0.2176620580148511, 'weight_decay': 0.1}


C:\Users\Tylus\PycharmProjects\QF634_Research_Methods\.venv1\lib\site-packages\transformers\training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
C:\Users\Tylus\AppData\Local\Temp\ipykernel_16292\3120888993.py:92: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.697700,0.693407,0.509105,0.509105,1.000000,0.674711
2,0.691800,0.694313,0.509105,0.509105,1.000000,0.674711


In [ ]:
# Use the best model to predict on test set
finbert_best_model = BertForSequenceClassification.from_pretrained('./finbert_best_model').to(torch.device('cuda'))

def predict_with_threshold(model, dataset, threshold):
    trainer = Trainer(model=model, tokenizer=tokenizer)
    predictions_output = trainer.predict(dataset)
    predictions = predictions_output.predictions
    labels = predictions_output.label_ids  # True labels from the dataset
    predicted_probabilities = torch.softmax(torch.tensor(predictions), dim=1).numpy()
    predicted_labels = (predicted_probabilities[:, 1] >= threshold).astype(int)

    return predicted_probabilities, predicted_labels, labels

# Use the best threshold for predictions
best_threshold = best_params['threshold']
finbert_probabilities, finbert_predicted_labels, test_labels = predict_with_threshold(finbert_best_model, test_tokenized_df, best_threshold)

# Compute evaluation metrics on the test set
precision, recall, f1, _ = precision_recall_fscore_support(test_labels, finbert_predicted_labels, average='binary')
accuracy = accuracy_score(test_labels, finbert_predicted_labels)

print(f"Test Accuracy: {accuracy}")
print(f"Test Precision: {precision}")
print(f"Test Recall: {recall}")
print(f"Test F1-Score: {f1}")

# Add predictions to the results dataframe
test_df_results['finbert_sentiment_label'] = finbert_predicted_labels
test_df_results

In [13]:
# Save predictions to the test dataframe
test_df['predictions'] = test_predictions
test_df.to_csv("test_predictions.csv", index=False)

C:\Users\Tylus\AppData\Local\Temp\ipykernel_12020\997101882.py:5: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(model=model, tokenizer=tokenizer)
